In [32]:
import requests
import pandas as pd
from datetime import datetime


In [33]:
all_records = []
start_year = datetime.now().year - 5
end_year = datetime.now().year

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"    

In [34]:
all_records = []

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }

        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"⚠️ Failed for {start_date}: {response.text[:200]}")
            continue

        try:
            data = response.json()
        except Exception as e:
            print(f"⚠️ JSON error for {start_date}: {e}")
            continue

        for f in data["features"]:
            p = f["properties"]
            g = f["geometry"]["coordinates"]
            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms"),
                "updated": pd.to_datetime(p.get("updated"), unit="ms"),
                "latitude": g[1] if g else None,
                "longitude": g[0] if g else None,
                "depth_km": g[2] if g else None,
                "mag": p.get("mag"),
                "magType": p.get("magType"),
                "place": p.get("place"),
                "type": p.get("type"),
                "status": p.get("status"),
                "tsunami": p.get("tsunami"),
                "sig": p.get("sig"),
                "net": p.get("net"),
                "depthError": p.get("depthError"),
                "ids": p.get("ids"),
                "sources": p.get("sources"),
                "types": p.get("types"),
                "nst": p.get("nst"),    
                "dmin": p.get("dmin"),
                "rms": p.get("rms"),    
                "gap": p.get("gap"),
                "magError": p.get("magError"),
                "magNst": p.get("magNst"),
                "locationSource": p.get("locationSource"),
                "magSource": p.get("magSource"),
            })


df = pd.DataFrame(all_records)

print("\n--- Data Pipeline Gathering Finished ---")
print(f"Total Rows Extracted: {df.shape[0]}")
print(f"Total Columns Staged: {df.shape[1]}")


            


--- Data Pipeline Gathering Finished ---
Total Rows Extracted: 119358
Total Columns Staged: 26


In [39]:
print(df.columns.tolist())
print(df.shape)

['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag', 'magType', 'place', 'type', 'status', 'tsunami', 'sig', 'net', 'depthError', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms', 'gap', 'magError', 'magNst', 'locationSource', 'magSource']
(119358, 26)


In [40]:
df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,type,...,sources,types,nst,dmin,rms,gap,magError,magNst,locationSource,magSource
0,us6000ddi8,2021-01-31 23:20:49.923,2021-04-16 19:02:44.040,-31.7493,-68.9337,17.27,4.7,mwr,"29 km SW of Villa Basilio Nievas, Argentina",earthquake,...,",us,",",dyfi,moment-tensor,origin,phase-data,",NaN,0.294,0.82,42.0,None,None,None,None
1,us6000dev6,2021-01-31 23:08:17.161,2021-04-16 19:03:47.040,-15.4902,-177.2052,426.71,4.1,mb,Fiji region,earthquake,...,",us,",",origin,phase-data,",NaN,1.471,0.29,64.0,None,None,None,None
2,us6000dev5,2021-01-31 22:54:19.760,2021-04-16 19:03:47.040,19.7529,121.3159,46.73,4.7,mb,"103 km SW of Basco, Philippines",earthquake,...,",us,",",origin,phase-data,",NaN,3.057,0.69,106.0,None,None,None,None
3,us6000ddhs,2021-01-31 22:06:00.832,2021-04-16 19:02:43.040,28.1524,57.2570,10.00,4.9,mb,"114 km N of M?n?b, Iran",earthquake,...,",us,",",origin,phase-data,",NaN,3.330,0.61,71.0,None,None,None,None
4,us6000dev4,2021-01-31 21:51:14.016,2021-04-16 19:03:46.040,71.3212,-3.7578,10.00,4.0,mb,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",earthquake,...,",us,",",origin,phase-data,",NaN,6.023,0.50,65.0,None,None,None,None


In [41]:
print(response.json())


{'type': 'FeatureCollection', 'metadata': {'generated': 1789334331000, 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-12-01&endtime=2027-01-01&minmagnitude=3', 'title': 'USGS Earthquakes', 'status': 200, 'api': '2.7.0', 'count': 0}, 'features': []}


In [42]:
df.to_csv("earthquake_raw_data.csv", index=False)

print("Raw data saved successfully!")

Raw data saved successfully!
